In [28]:
#from google.colab import drive
#drive.mount('/content/drive')

In [29]:
#數據overview
import pandas as pd
raw_data = pd.read_excel("model_train_data.xlsx")
print("=" * 60)
print("【所有欄位名稱及數據類型】")
print("=" * 60)
for col, dtype in raw_data.dtypes.items():
    print(f"  {col:<40} {dtype}")
print("\n" + "=" * 60)
print("【非數字類型欄位的類別分佈】")
print("=" * 60)
non_numeric_cols = raw_data.select_dtypes(exclude=["number"]).columns
if len(non_numeric_cols) == 0:
    print("  （無非數字類型的欄位）")
else:
    for col in non_numeric_cols:
        print(f"\n▶ 欄位：{col}（類型：{raw_data[col].dtype}）")
        print("-" * 40)
        value_counts = raw_data[col].value_counts(dropna=False)
        for val, count in value_counts.items():
            print(f"    {str(val):<30} 數量：{count}")
        print(f"  → 共 {len(value_counts)} 種類別，{raw_data[col].notna().sum()} 個非空值")

【所有欄位名稱及數據類型】
  Due Date                                 datetime64[ns]
  Ship From                                object
  Ship Via                                 object
  Vessel Name                              object
  Vendor Name                              object
  Incoterm                                 object
  PO Num                                   object
  Cust Num                                 object
  Season                                   object
  Buy No                                   object
  Item                                     object
  RMA Description                          object
  UM                                       object
  Qty                                      float64
  Unit Price With Surcharge                float64
  Total Material Cost (Baht)               float64
  Exwork (Baht)                            int64
  %Exwork (M/L)                            float64
  Freight Cost (Baht)                      int64
  %Freight (O/L)          

In [30]:
column_mapping = {
    'Due Date': 'Date',
    'Ship From': 'Ship_From',
    'Ship Via': 'Ship_Via',
    'Vessel Name': 'Vessel_Name',
    'Vendor Name': 'Vendor_Name',
    'Incoterm': 'Incoterm',
    'PO Num': 'PO_Num',
    'Cust Num': 'Cust_Num',
    'Season': 'Season',
    'Buy No': 'Buy_No',
    'Item': 'Item',
    'Unit Price With Surcharge': 'Unit_Price',
    'Qty': 'Qty',
    'Total Material Cost (Baht)': 'Total_Material_Cost',
    
    'destination': 'Destination',
    'Ship Via Category': 'Ship_Via_Cat',
    
    'Exwork (Baht)': 'Exwork(M)',
    '%Exwork (M/L)': 'P_Exwork(M)',
    'Freight Cost (Baht)': 'Freight(O)',
    '%Freight (O/L)': 'P_Freight(O)',
    'Local (Baht)': 'Local(Q)',
    '%Local (Q/L)': 'P_Local(Q)',
    'Brokerage (Baht)': 'Brokerage(S)',
    '%Brokerage (S/L)': 'P_Brokerage(S)',
    'Total Import cost (Baht) (M+O+Q+S)': 'Total_Import_cost(U)',
    '% Import Cost (U/L)': 'P_Import_Cost(U)'
}

raw_data = raw_data.rename(columns=column_mapping)
print(raw_data.columns.tolist())

['Date', 'Ship_From', 'Ship_Via', 'Vessel_Name', 'Vendor_Name', 'Incoterm', 'PO_Num', 'Cust_Num', 'Season', 'Buy_No', 'Item', 'RMA Description', 'UM', 'Qty', 'Unit_Price', 'Total_Material_Cost', 'Exwork(M)', 'P_Exwork(M)', 'Freight(O)', 'P_Freight(O)', 'Local(Q)', 'P_Local(Q)', 'Brokerage(S)', 'P_Brokerage(S)', 'Total_Import_cost(U)', 'P_Import_Cost(U)', 'Ship_Via_Cat', 'Destination']


In [31]:
features = ['Ship_From', 'Ship_Via', 'Vendor_Name', 'Incoterm', 'Item',
            'Total_Material_Cost', 'Exwork(M)', 'Freight(O)',
            'Local(Q)', 'Brokerage(S)', 'Total_Import_cost(U)', 'Unit_Price','Qty']######
df = raw_data[features].copy()
#print("size:", df.shape)
#print(df.isnull().sum())
df = df.dropna()
#print("===/n")
#print("size:", df.shape)
#print(df.isnull().sum())

In [32]:
df['Ship_From_Via'] = df['Ship_From'] + '_' + df['Ship_Via']
df['Vendor_From_Via'] = df['Vendor_Name'] + '_' + df['Ship_From_Via']
df['Vendor_Item'] = df['Vendor_Name'] + '_' + df['Item']

print("Ship_From_Via:")
print(df['Ship_From_Via'].unique())
print("\nVendor_From_Via:")
print(df['Vendor_From_Via'].unique())
print("\nShip_From_Via size:", df['Ship_From_Via'].nunique())
print("Vendor_From_Via size:", df['Vendor_From_Via'].nunique())

Ship_From_Via:
['TAIWAN KEELUNG_SEA' 'HK_SEA' 'JAPAN OSAKA_SEA' 'KOREA BUSAN_SEA'
 'TAIWAN KEELUNG_DHL' 'TAIWAN KEELUNG_FED' 'TAIWAN KEELUNG_AIR' 'HK_DHL'
 'HK_FED' 'JAPAN OSAKA_FED' 'CHINA SHANGHAI_DHL' 'CHINA SHANGHAI_SEA'
 'VIETNAM_FED' 'VIETNAM_SEA' 'JAPAN OSAKA_DHL' 'SWEDEN_SEA'
 'CHINA SHENZHEN_SEA' 'ITALY_SEA' 'JAPAN OSAKA_AIR' 'TAIWAN KAOHSIUNG_SEA'
 'KOREA BUSAN_AIR' 'VIETNAM_DHL' 'CHINA SHANGHAI_FED' 'KOREA BUSAN_DHL'
 'KOREA BUSAN_FED' 'CHINA SHENZHEN_DHL' 'ITALY_FED' 'ITALY_AIR'
 'CHINA XIAMEN_DHL' 'CHINA SHANGHAI_AIR' 'NEW ZELAND_AIR' 'VIETNAM_AIR']

Vendor_From_Via:
['KINGWHALE CORPORATION_TAIWAN KEELUNG_SEA'
 'AVERY DENNISON HONG  KONG   B.V._HK_SEA'
 'ASAHI KASEI ADVANCE CORPORATION_JAPAN OSAKA_SEA'
 'TS COMPANY Co., Ltd._KOREA BUSAN_SEA'
 'E.TEXTINT CORP._TAIWAN KEELUNG_DHL'
 'GOLDENGOOSE GROUP INC._TAIWAN KEELUNG_FED'
 'Universal Trim Supply Co.,Ltd._TAIWAN KEELUNG_AIR'
 'AVERY DENNISON PAXAR (CHINA) LIMITED_HK_DHL'
 'YAGI & CO.,LTD._JAPAN OSAKA_SEA'
 'UNITEX INTERNAT

In [33]:
df['Exwork_is_zero']    = (df['Exwork(M)'] == 0).astype(int)
df['Freight_is_zero']   = (df['Freight(O)'] == 0).astype(int)
df['Local_is_zero']     = (df['Local(Q)'] == 0).astype(int)
df['Brokerage_is_zero'] = (df['Brokerage(S)'] == 0).astype(int)

In [34]:
import numpy as np
num_cols = ['Total_Material_Cost', 'Exwork(M)', 'Freight(O)',
            'Local(Q)', 'Brokerage(S)', 'Total_Import_cost(U)', 'Unit_Price']
df[num_cols] = np.log1p(df[num_cols])
#print(df[num_cols].describe())

In [35]:
cat_cols = ['Vendor_From_Via', 'Incoterm', 'Item']
for col in cat_cols:
    df[col] = df[col].astype('category')

In [36]:
from sklearn.model_selection import train_test_split

X = df[['Vendor_From_Via', 'Incoterm', 'Item', 'Total_Material_Cost', 'Unit_Price',
        'Exwork_is_zero', 'Freight_is_zero', 'Local_is_zero', 'Brokerage_is_zero']]
targets = ['Exwork(M)', 'Freight(O)', 'Local(Q)', 'Brokerage(S)']

X_train, X_test, y_train, y_test = train_test_split(X, df[targets], test_size=0.2, random_state=42)



In [37]:
import xgboost as xgb
models = {}
for target in targets:
    model = xgb.XGBRegressor(enable_categorical=True, random_state=42)
    model.fit(X_train, y_train[target])
    models[target] = model

In [38]:
#incoterm rules //小量提升了
INCOTERM_RULES = {
    'EXW': [],
    'FCA': ['Exwork(M)'],
    'FOB': ['Exwork(M)'],
    'CFR': ['Exwork(M)', 'Freight(O)'],
    'CIF': ['Exwork(M)', 'Freight(O)'],
    'CPT': ['Exwork(M)', 'Freight(O)'],
    'DAP': ['Exwork(M)', 'Freight(O)', 'Local(Q)'],
    'DDP': ['Exwork(M)', 'Freight(O)', 'Local(Q)', 'Brokerage(S)'],
}

predictions = {}
for target in targets:
    predictions[target] = np.expm1(models[target].predict(X_test)).clip(min=0)

# DHL/FED rules //only微量提升
for i, idx in enumerate(X_test.index):
    incoterm = df.loc[idx, 'Incoterm']
    ship_via = str(df.loc[idx, 'Ship_Via']).upper()
    for col in INCOTERM_RULES.get(incoterm, []):
        predictions[col][i] = 0.0
    if any(k in ship_via for k in ['DHL', 'FED']):
        predictions['Exwork(M)'][i] = 0.0


predictions['Total_Import_cost(U)'] = sum(predictions[target] for target in targets)

In [39]:
from sklearn.metrics import mean_absolute_error, r2_score
for target in targets:
    y_true = np.expm1(y_test[target])
    y_pred = predictions[target]

    print(f"\n{target}:")
    print(f"  MAE: {mean_absolute_error(y_true, y_pred):.2f}")
    print(f"  R²:  {r2_score(y_true, y_pred):.4f}")


Exwork(M):
  MAE: 138.68
  R²:  0.7442

Freight(O):
  MAE: 737.71
  R²:  0.9231

Local(Q):
  MAE: 240.44
  R²:  0.8860

Brokerage(S):
  MAE: 374.85
  R²:  0.8758


In [40]:
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

X_cv = X
targets_cv = targets
y_cv = df[targets_cv]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = {t: {'r2': [], 'mae': []} for t in targets_cv}

for fold, (train_idx, test_idx) in enumerate(kf.split(X_cv)):
    X_train_fold = X_cv.iloc[train_idx]
    X_test_fold  = X_cv.iloc[test_idx]
    y_train_fold = y_cv.iloc[train_idx]
    y_test_fold  = y_cv.iloc[test_idx]

    models_fold = {}
    for target in targets_cv:
        model = xgb.XGBRegressor(enable_categorical=True, random_state=42)
        model.fit(X_train_fold, y_train_fold[target])
        models_fold[target] = model
    for target in targets_cv:
        y_pred_log = models_fold[target].predict(X_test_fold)
        y_pred = np.expm1(y_pred_log).clip(min=0)
        y_true = np.expm1(y_test_fold[target].values)

        r2 = r2_score(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        scores[target]['r2'].append(r2)
        scores[target]['mae'].append(mae)

print("===== 5-Fold Cross-Validation Results =====")
for target in targets_cv:
    r2_mean = np.mean(scores[target]['r2'])
    r2_std  = np.std(scores[target]['r2'])
    mae_mean = np.mean(scores[target]['mae'])
    mae_std  = np.std(scores[target]['mae'])
    print(f"\n{target}:")
    print(f"  R²  : {r2_mean:.4f} (±{r2_std:.4f})")
    print(f"  MAE : {mae_mean:.2f} (±{mae_std:.2f})")

===== 5-Fold Cross-Validation Results =====

Exwork(M):
  R²  : 0.6273 (±0.1557)
  MAE : 178.67 (±39.76)

Freight(O):
  R²  : 0.7235 (±0.2661)
  MAE : 1194.17 (±435.66)

Local(Q):
  R²  : 0.8559 (±0.0338)
  MAE : 275.02 (±31.25)

Brokerage(S):
  R²  : 0.8536 (±0.0360)
  MAE : 426.06 (±41.78)


In [41]:
# model2
df['Vendor_Item'] = df['Vendor_Item'].astype('category')
mask_exw = (df['Incoterm'] == 'EXW') & (df['Exwork_is_zero'] == 0)
df_exw = df[mask_exw].copy()
X_exw = df_exw[['Vendor_Item', 'Total_Material_Cost', 'Vendor_From_Via', 'Item', 'Unit_Price']]
y_exw = df_exw['Exwork(M)']
X_train_exw, X_test_exw, y_train_exw, y_test_exw = train_test_split(
    X_exw, y_exw, test_size=0.2, random_state=42)

model2 = xgb.XGBRegressor(enable_categorical=True, random_state=42)
model2.fit(X_train_exw, y_train_exw)

y_pred_exw = np.expm1(model2.predict(X_test_exw)).clip(min=0)
y_true_exw = np.expm1(y_test_exw)

print(f"Exwork model2 R²:  {r2_score(y_true_exw, y_pred_exw):.4f}")
print(f"Exwork model2 MAE: {mean_absolute_error(y_true_exw, y_pred_exw):.2f}")


Exwork model2 R²:  -0.0195
Exwork model2 MAE: 933.49


In [42]:
#model3
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder

mask_exw = (df['Incoterm'] == 'EXW') & (df['Exwork_is_zero'] == 0)
df_exw = df[mask_exw].copy()

le_vendor_item = LabelEncoder()
le_vendor = LabelEncoder()
le_item = LabelEncoder()

df_exw['Vendor_Item_enc']    = le_vendor_item.fit_transform(df_exw['Vendor_Item'].astype(str))
df_exw['Vendor_From_Via_enc'] = le_vendor.fit_transform(df_exw['Vendor_From_Via'].astype(str))
df_exw['Item_enc']            = le_item.fit_transform(df_exw['Item'].astype(str))

X_exw = df_exw[['Vendor_Item_enc', 'Vendor_From_Via_enc', 'Item_enc', 'Total_Material_Cost']].values
y_exw = df_exw['Exwork(M)'].values

X_train_exw, X_test_exw, y_train_exw, y_test_exw = train_test_split(
    X_exw, y_exw, test_size=0.2, random_state=42)

#轉tensor
X_train_t = torch.FloatTensor(X_train_exw)
y_train_t = torch.FloatTensor(y_train_exw).unsqueeze(1)
X_test_t  = torch.FloatTensor(X_test_exw)

class ExworkNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

model3 = ExworkNet(X_train_exw.shape[1])
optimizer = torch.optim.Adam(model3.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

for epoch in range(2000):
    model3.train()
    optimizer.zero_grad()
    pred = model3(X_train_t)
    loss = loss_fn(pred, y_train_t)
    loss.backward()
    optimizer.step()


model3.eval()
with torch.no_grad():
    y_pred_log = model3(X_test_t).squeeze().numpy()

y_pred_exw = np.expm1(y_pred_log).clip(min=0)
y_true_exw = np.expm1(y_test_exw)

print(f"神經網絡 Exwork R²:  {r2_score(y_true_exw, y_pred_exw):.4f}")
print(f"神經網絡 Exwork MAE: {mean_absolute_error(y_true_exw, y_pred_exw):.2f}")

神經網絡 Exwork R²:  0.8482
神經網絡 Exwork MAE: 636.12


In [43]:
y_true_total = np.expm1(y_test['Exwork(M)']) + np.expm1(y_test['Freight(O)']) + \
               np.expm1(y_test['Local(Q)']) + np.expm1(y_test['Brokerage(S)'])
y_pred_total = predictions['Total_Import_cost(U)']

print(f"\nTotal_Import_cost(U):")
print(f"  MAE: {mean_absolute_error(y_true_total, y_pred_total):.2f}")
print(f"  R²:  {r2_score(y_true_total, y_pred_total):.4f}")


Total_Import_cost(U):
  MAE: 1350.21
  R²:  0.9238


In [44]:
pred_freight  = predictions['Freight(O)']
pred_local    = predictions['Local(Q)']
pred_brokerage= predictions['Brokerage(S)']
y_true_total = np.expm1(y_test['Exwork(M)']) + np.expm1(y_test['Freight(O)']) + \
               np.expm1(y_test['Local(Q)'])   + np.expm1(y_test['Brokerage(S)'])

#A:全用model1
pred_total_A = predictions['Exwork(M)'] + pred_freight + pred_local + pred_brokerage
print("A：全用model1")
print(f"  MAE: {mean_absolute_error(y_true_total, pred_total_A):.2f}")
print(f"  R²:  {r2_score(y_true_total, pred_total_A):.4f}")

#B：Exwork用 model2 (XGBoost)，其他model1
pred_exw_B = np.zeros(len(X_test))
mask_exw_test = (df.loc[X_test.index, 'Incoterm'] == 'EXW') & \
                (df.loc[X_test.index, 'Exwork_is_zero'] == 0)
X_test_exw_B = df.loc[X_test.index[mask_exw_test], ['Vendor_Item', 'Total_Material_Cost', 'Vendor_From_Via', 'Item', 'Unit_Price']]
pred_exw_B[mask_exw_test.values] = np.expm1(model2.predict(X_test_exw_B)).clip(min=0)

pred_total_B = pred_exw_B + pred_freight + pred_local + pred_brokerage
print("\nB：only Exwork用 model2")
print(f"  MAE: {mean_absolute_error(y_true_total, pred_total_B):.2f}")
print(f"  R²:  {r2_score(y_true_total, pred_total_B):.4f}")

#C：Exwork用model3(神經網絡)，其他model1
pred_exw_C = np.zeros(len(X_test))
df_exw_test_C = df.loc[X_test.index[mask_exw_test]].copy()
df_exw_test_C['Vendor_Item_enc']     = le_vendor_item.transform(df_exw_test_C['Vendor_Item'].astype(str))
df_exw_test_C['Vendor_From_Via_enc'] = le_vendor.transform(df_exw_test_C['Vendor_From_Via'].astype(str))
df_exw_test_C['Item_enc']            = le_item.transform(df_exw_test_C['Item'].astype(str))

X_test_t_C = torch.FloatTensor(
    df_exw_test_C[['Vendor_Item_enc', 'Vendor_From_Via_enc', 'Item_enc', 'Total_Material_Cost']].values)
model3.eval()
with torch.no_grad():
    pred_exw_C[mask_exw_test.values] = np.expm1(model3(X_test_t_C).squeeze().numpy()).clip(min=0)

pred_total_C = pred_exw_C + pred_freight + pred_local + pred_brokerage
print("\nC：only Exwork用model3")
print(f"  MAE: {mean_absolute_error(y_true_total, pred_total_C):.2f}")
print(f"  R²:  {r2_score(y_true_total, pred_total_C):.4f}")

A：全用model1
  MAE: 1350.21
  R²:  0.9238

B：only Exwork用 model2
  MAE: 1296.80
  R²:  0.9255

C：only Exwork用model3
  MAE: 1345.76
  R²:  0.9243


In [45]:
# predict_import_cost function的限制
AIR_SHIP_FROMS = [
    'HK BY AIR', 'TAIWAN KEELUNG BY AIR', 'JAPAN OSAKA BY AIR',
    'VIETNAM BY AIR', 'CHINA SHANGHAI BY AIR', 'JAPAN OSAKA TO MM BY AIR',
    'ITALY BY AIR', 'KOREA BUSAN BY AIR', 'TAIWAN KEELUNG TO MM BY AIR',
    'HK TO MM BY AIR', 'FRANCE BY AIR', 'FRANCE TO MM BY AIR',
    'CHINA GUANGZHOU BY AIR', 'KOREA BUSAN TO MM BY AIR', 'CHINA XIAMEN BY AIR',
    'CHINA SHANGHAI TO MM BY AIR', 'VIETNAM TO MM BY AIR', 'NEW ZELAND BY AIR',
    'TAIWAN KAOHSIUNG TO MM BY AIR', 'THAILAND TO MM BY AIR',
    'TAIWAN KAOHSIUNG BY AIR', 'CHINA SHENZHEN BY AIR'
]

INVALID_INCOTERM_VIA = [
    ('FOB', 'DHL'), ('FOB', 'FED'),
    ('CIF', 'FED'),
    ('DAP', 'DHL'), ('DAP', 'FED'),
]

def is_valid_combination(ship_from, ship_via, incoterm):
    ship_via_upper = ship_via.upper()
    # 規則1：BY AIR的ship_from只能配AIR/DHL/FED
    if ship_from in AIR_SHIP_FROMS:
        if not any(k in ship_via_upper for k in ['AIR', 'DHL', 'FED']):
            return False, f"'{ship_from}' 只能配 AIR/DHL/FED，不能配 '{ship_via}'"
    # 規則2：非法組合of Incoterm + Ship_Via
    for inv_inc, inv_via in INVALID_INCOTERM_VIA:
        if incoterm == inv_inc and inv_via in ship_via_upper:
            return False, f"'{incoterm}' 不能與 '{ship_via}' 搭配"
    # 規則345678。。。？

    return True, None

In [46]:
#輸入：vendor_name, ship_from, ship_via, item, total_material_cost, incoterm
#輸出：predict的Total_Import_cost
def predict_import_cost(vendor_name, ship_from, ship_via, item, total_material_cost, unit_price, incoterm, use_model='B'):



    valid, reason = is_valid_combination(ship_from, ship_via, incoterm)
    if not valid:
        print(f"Warning: Unreasonable combination in input! Reason：{reason}")
        choice = input("Enter 'y' to ignore the warning and continue, any other key to exit:").strip().lower()
        if choice != 'y':
            print("Exited.")
            return None



    ship_from_via   = f"{ship_from}_{ship_via}"
    vendor_from_via = f"{vendor_name}_{ship_from_via}"
    vendor_item     = f"{vendor_name}_{item}"

    zero_cols = INCOTERM_RULES.get(incoterm, [])
    exwork_is_zero   = 1 if 'Exwork(M)'    in zero_cols else 0
    freight_is_zero  = 1 if 'Freight(O)'   in zero_cols else 0
    local_is_zero    = 1 if 'Local(Q)'     in zero_cols else 0
    brokerage_is_zero= 1 if 'Brokerage(S)' in zero_cols else 0

    input_m1 = pd.DataFrame({
        'Vendor_From_Via':  [vendor_from_via],
        'Incoterm':         [incoterm],
        'Item':             [item],
        'Total_Material_Cost': [np.log1p(total_material_cost)],
        'Unit_Price':       [np.log1p(unit_price)],
        'Exwork_is_zero':   [exwork_is_zero],
        'Freight_is_zero':  [freight_is_zero],
        'Local_is_zero':    [local_is_zero],
        'Brokerage_is_zero':[brokerage_is_zero],
    })
    for col in ['Vendor_From_Via', 'Incoterm', 'Item']:
        input_m1[col] = input_m1[col].astype('category')

    # 預測 Freight, Local, Brokerage（用model1）
    results = {}
    for target in ['Freight(O)', 'Local(Q)', 'Brokerage(S)']:
        results[target] = np.expm1(models[target].predict(input_m1))[0].clip(min=0)

    # 預測 Exwork（用2，3）
    if exwork_is_zero == 1:
        results['Exwork(M)'] = 0.0
    elif use_model == 'A':
        results['Exwork(M)'] = np.expm1(models['Exwork(M)'].predict(input_m1))[0].clip(min=0)
    elif use_model == 'B':
        input_m2 = pd.DataFrame({
            'Vendor_Item':        [vendor_item],
            'Total_Material_Cost':[np.log1p(total_material_cost)],
            'Vendor_From_Via':    [vendor_from_via],
            'Item':               [item],
            'Unit_Price':         [np.log1p(unit_price)]
        })
        for col in ['Vendor_Item', 'Vendor_From_Via', 'Item']:
            input_m2[col] = input_m2[col].astype('category')
        results['Exwork(M)'] = np.expm1(model2.predict(input_m2))[0].clip(min=0)
    elif use_model == 'C':
        enc_vi  = le_vendor_item.transform([vendor_item])[0]
        enc_vfv = le_vendor.transform([vendor_from_via])[0]
        enc_i   = le_item.transform([item])[0]
        x_t = torch.FloatTensor([[enc_vi, enc_vfv, enc_i, np.log1p(total_material_cost)]])
        model3.eval()
        with torch.no_grad():
            results['Exwork(M)'] = np.expm1(model3(x_t).item()).clip(min=0)


    for col in zero_cols:
        results[col] = 0.0

    results['Total_Import_cost(U)'] = sum(results[target] for target in targets)

    for key, val in results.items():
        print(f"{key}: {val:.2f} Baht")
    return results

In [47]:
predict_import_cost(
    vendor_name='KINGWHALE CORPORATION',
    ship_from='TAIWAN KEELUNG TO MM',
    ship_via='SEA',
    item='CKN',
    total_material_cost=776889,
    unit_price=10000,
    incoterm='FOB',
    use_model='B'
)



Freight(O): 275.11 Baht
Local(Q): 1041.76 Baht
Brokerage(S): 2089.14 Baht
Exwork(M): 0.00 Baht
Total_Import_cost(U): 3406.01 Baht


{'Freight(O)': 275.1111145019531,
 'Local(Q)': 1041.7576904296875,
 'Brokerage(S)': 2089.14453125,
 'Exwork(M)': 0.0,
 'Total_Import_cost(U)': 3406.0133361816406}

In [48]:
def find_best_combinations(
    item,                          # 必填
    total_material_cost=50000,     # 選填，預設50000
    unit_price=100.0,
    vendor_options=None,           # 以下都是選填
    ship_from_options=None,
    ship_via_options=None,
    incoterm_options=None,
    use_model='B',
    top_n=5
):
    all_incoterms = list(INCOTERM_RULES.keys())
    incoterms = incoterm_options if incoterm_options else all_incoterms

    # 取训练集里真实存在的 (Vendor, Ship_From, Ship_Via) 组合，再按用户输入过滤
    combo_df = df[['Vendor_Name', 'Ship_From', 'Ship_Via']].drop_duplicates()
    if vendor_options:
        combo_df = combo_df[combo_df['Vendor_Name'].isin(vendor_options)]
    if ship_from_options:
        combo_df = combo_df[combo_df['Ship_From'].isin(ship_from_options)]
    if ship_via_options:
        combo_df = combo_df[combo_df['Ship_Via'].isin(ship_via_options)]

    if ship_from_options and ship_via_options and incoterm_options:
        all_invalid = all(
            not is_valid_combination(sf, sv, inc)[0]
            for sf in combo_df['Ship_From'].unique()
            for sv in combo_df['Ship_Via'].unique()
            for inc in incoterms
        )
        if all_invalid:
            print("Warning: All combinations you entered are invalid. No feasible combination exists.")
            choice = input("Enter 'y' to ignore warning and continue, any other key to exit: ").strip().lower()
            if choice != 'y':
                print("Exiting program.")
                return
            else:
                print("Continuing...\n")

    results_list = []
    skipped_invalid = 0
    skipped_error = 0

    for _, row in combo_df.iterrows():
        v, sf, sv = row['Vendor_Name'], row['Ship_From'], row['Ship_Via']
        for inc in incoterms:
            valid, _ = is_valid_combination(sf, sv, inc)
            if not valid:
                skipped_invalid += 1
                continue
            try:
                r = predict_import_cost(
                    vendor_name=v, ship_from=sf, ship_via=sv,
                    item=item, total_material_cost=total_material_cost,
                    unit_price=unit_price, incoterm=inc, use_model=use_model
                )
                if r is None:
                    skipped_error += 1
                    continue
                results_list.append({
                    'Vendor_Name': v, 'Ship_From': sf, 'Ship_Via': sv, 'Incoterm': inc,
                    'Exwork(M)':   round(r['Exwork(M)'], 2),
                    'Freight(O)':  round(r['Freight(O)'], 2),
                    'Local(Q)':    round(r['Local(Q)'], 2),
                    'Brokerage(S)':round(r['Brokerage(S)'], 2),
                    'Total_Import_cost(U)': round(r['Total_Import_cost(U)'], 2),
                })
            except:
                skipped_error += 1

    print(f"Valid combinations: {len(results_list)}")
    print(f"Skipped due to restrictions: {skipped_invalid}")
    print(f"Skipped due to model errors: {skipped_error}")

    if not results_list:
        print("No valid combinations found.")
        return

    actual_top_n = min(top_n, len(results_list))
    if actual_top_n < top_n:
        print(f"Only {actual_top_n} valid combination(s) found, fewer than requested {top_n}.")

    results_df = pd.DataFrame(results_list)
    top_results = results_df.nsmallest(actual_top_n, 'Total_Import_cost(U)').reset_index(drop=True)

    print(f"\n=== Top {actual_top_n} combinations with smallest Total_Import_cost ===")
    print(f"Item: {item}, Total_Material_Cost: {total_material_cost} Baht\n")
    for i, row in top_results.iterrows():
        print(f"--- Option {i+1} ---")
        print(f"  Vendor:    {row['Vendor_Name']}")
        print(f"  Ship_From: {row['Ship_From']}")
        print(f"  Ship_Via:  {row['Ship_Via']}")
        print(f"  Incoterm:  {row['Incoterm']}")
        print(f"  Exwork(M):         {row['Exwork(M)']:>10.2f} Baht")
        print(f"  Freight(O):        {row['Freight(O)']:>10.2f} Baht")
        print(f"  Local(Q):          {row['Local(Q)']:>10.2f} Baht")
        print(f"  Brokerage(S):      {row['Brokerage(S)']:>10.2f} Baht")
        print(f"  Total_Import_cost: {row['Total_Import_cost(U)']:>10.2f} Baht\n")

    return top_results

In [49]:
#function testing
find_best_combinations(
    item='Knitted fabric',
    total_material_cost=776889,
    incoterm_options=['FOB', 'EXW', 'CIF'],
    vendor_options=['KINGWHALE CORPORATION', 'AVERY DENNISON HONG  KONG   B.V.'],
    ship_from_options=['HK', 'TAIWAN KEELUNG TO MM'],
    ship_via_options=['SEA', 'DHL'],
)

Valid combinations: 0
Skipped due to restrictions: 1
Skipped due to model errors: 5
No valid combinations found.


In [50]:
"""
def main():
    print("=== Import Cost Optimization Tool ===\n")

    # Global df must be loaded from previous notebook cells
    global df
    try:
        available_items = sorted(df['Item'].dropna().unique())
        available_vendors = set(df['Vendor_Name'].dropna().unique())
        available_ship_from = set(df['Ship_From'].dropna().unique())
        available_ship_via = set(df['Ship_Via'].dropna().unique())
        valid_incoterms = list(INCOTERM_RULES.keys())
    except NameError:
        print("Error: Dataset 'df' or INCOTERM_RULES not found. Please run the prerequisite cells.")
        return

    # 1. Item (required, must exist in dataset)
    while True:
        item = input("Enter Item name (required): ").strip()
        if not item:
            print("Item cannot be empty. Please re-enter.")
            continue
        if item not in available_items:
            print(f"'{item}' is not in the available Item list. Available items (first 10): {available_items[:10]}...")
            continue
        break

    # 2. Total Material Cost (positive number, default 50000)
    while True:
        cost_input = input("Enter Total Material Cost (default 50000): ").strip()
        if cost_input == "":
            total_material_cost = 50000
            break
        try:
            val = float(cost_input)
            if val < 0:
                raise ValueError
            total_material_cost = val
            break
        except ValueError:
            print("Please enter a valid positive number, or leave blank to use default.")
    # new: Unit Price With Surcharge (positive number, default 100)
    while True:
        unit_input = input("Enter Unit Price With Surcharge (default 100): ").strip()
        if unit_input == "":
            unit_price = 100.0
            break
        try:
            val = float(unit_input)
            if val <= 0:
                raise ValueError
            unit_price = val
            break
        except ValueError:
            print("Please enter a valid positive number, or leave blank to use default.")

    # 3. Vendor options (optional, each must exist if provided)
    while True:
        vendor_input = input("Enter Vendor name(s), separated by commas (leave blank for all): ").strip()
        if vendor_input == "":
            vendor_options = None
            break
        vendors = [v.strip() for v in vendor_input.split(",") if v.strip()]
        invalid = [v for v in vendors if v not in available_vendors]
        if invalid:
            print(f"The following Vendor(s) do not exist: {invalid}")
            print("Please re-enter valid Vendor names (or leave blank for all).")
        else:
            vendor_options = vendors
            break

    # 4. Ship From options
    while True:
        sf_input = input("Enter Ship From location(s), separated by commas (leave blank for all): ").strip()
        if sf_input == "":
            ship_from_options = None
            break
        sf_list = [s.strip() for s in sf_input.split(",") if s.strip()]
        invalid = [s for s in sf_list if s not in available_ship_from]
        if invalid:
            print(f"The following Ship From location(s) do not exist: {invalid}")
            print("Please re-enter valid locations (or leave blank for all).")
        else:
            ship_from_options = sf_list
            break

    # 5. Ship Via options
    while True:
        sv_input = input("Enter Ship Via method(s), separated by commas (leave blank for all): ").strip()
        if sv_input == "":
            ship_via_options = None
            break
        sv_list = [v.strip() for v in sv_input.split(",") if v.strip()]
        invalid = [v for v in sv_list if v not in available_ship_via]
        if invalid:
            print(f"The following Ship Via method(s) do not exist: {invalid}")
            print("Please re-enter valid methods (or leave blank for all).")
        else:
            ship_via_options = sv_list
            break

    # 6. Incoterm options
    while True:
        inc_input = input(f"Enter Incoterm(s), separated by commas (leave blank for all, options: {valid_incoterms}): ").strip()
        if inc_input == "":
            incoterm_options = None
            break
        inc_list = [i.strip().upper() for i in inc_input.split(",") if i.strip()]
        invalid = [i for i in inc_list if i not in valid_incoterms]
        if invalid:
            print(f"Unsupported Incoterm(s): {invalid}. Supported: {valid_incoterms}")
            print("Please re-enter.")
        else:
            incoterm_options = inc_list
            break

    # 7. Model (A/B/C)
    while True:
        model_input = input("Choose Exwork prediction model (A/B/C, default B): ").strip().upper()
        if model_input == "":
            use_model = 'B'
            break
        if model_input in ['A', 'B', 'C']:
            use_model = model_input
            break
        print("Invalid model option. Please enter A, B, or C (or leave blank for default B).")

    # 8. Top N
    while True:
        top_input = input("Enter number of top combinations to display (default 5): ").strip()
        if top_input == "":
            top_n = 5
            break
        try:
            top_n = int(top_input)
            if top_n <= 0:
                raise ValueError
            break
        except ValueError:
            print("Please enter a positive integer, or leave blank for default 5.")


    print("\n--- Parameter Settings ---")
    print(f"Item: {item}")
    print(f"Total Material Cost: {total_material_cost}")
    print(f"Unit Price: {unit_price}")
    print(f"Vendor options: {vendor_options if vendor_options else 'All'}")
    print(f"Ship From options: {ship_from_options if ship_from_options else 'All'}")
    print(f"Ship Via options: {ship_via_options if ship_via_options else 'All'}")
    print(f"Incoterm options: {incoterm_options if incoterm_options else 'All'}")
    print(f"Model: {use_model}")
    print(f"Top N: {top_n}")
    print("----------------\n")

    try:
        result_df = find_best_combinations(
            item=item,
            total_material_cost=total_material_cost,
            vendor_options=vendor_options,
            ship_from_options=ship_from_options,
            ship_via_options=ship_via_options,
            incoterm_options=incoterm_options,
            use_model=use_model,
            top_n=top_n
        )
        if result_df is None or result_df.empty:
            print("No valid combination found. Please adjust your criteria and try again.")
        else:
            print("\n=== Best Combinations Table ===")
            print(result_df.to_string(index=False))
    except Exception as e:
        print(f"Error during execution: {e}")
        print("Please check if the models are trained and input parameters are correct.")

if __name__ == "__main__":
    main()"""

'\ndef main():\n    print("=== Import Cost Optimization Tool ===\n")\n\n    # Global df must be loaded from previous notebook cells\n    global df\n    try:\n        available_items = sorted(df[\'Item\'].dropna().unique())\n        available_vendors = set(df[\'Vendor_Name\'].dropna().unique())\n        available_ship_from = set(df[\'Ship_From\'].dropna().unique())\n        available_ship_via = set(df[\'Ship_Via\'].dropna().unique())\n        valid_incoterms = list(INCOTERM_RULES.keys())\n    except NameError:\n        print("Error: Dataset \'df\' or INCOTERM_RULES not found. Please run the prerequisite cells.")\n        return\n\n    # 1. Item (required, must exist in dataset)\n    while True:\n        item = input("Enter Item name (required): ").strip()\n        if not item:\n            print("Item cannot be empty. Please re-enter.")\n            continue\n        if item not in available_items:\n            print(f"\'{item}\' is not in the available Item list. Available items (first

In [51]:
#model_MM和model_Thai
# ===== 分廠訓練準備：拆分 df_MM 和 df_Thai =====

features_raw = ['Ship_From', 'Ship_Via', 'Vendor_Name', 'Incoterm', 'Item',
                'Total_Material_Cost', 'Exwork(M)', 'Freight(O)',
                'Local(Q)', 'Brokerage(S)', 'Total_Import_cost(U)', 'Unit_Price', 'Qty']

df_MM   = raw_data[raw_data['Destination'] == 'MM'][features_raw].dropna().copy()
df_Thai = raw_data[raw_data['Destination'] == 'Thailand'][features_raw].dropna().copy()

def apply_fe(d):
    d = d.copy()
    # 組合特徵
    d['Ship_From_Via']   = d['Ship_From'] + '_' + d['Ship_Via']
    d['Vendor_From_Via'] = d['Vendor_Name'] + '_' + d['Ship_From_Via']
    d['Vendor_Item']     = d['Vendor_Name'] + '_' + d['Item']
    # zero flags
    d['Exwork_is_zero']    = (d['Exwork(M)'] == 0).astype(int)
    d['Freight_is_zero']   = (d['Freight(O)'] == 0).astype(int)
    d['Local_is_zero']     = (d['Local(Q)'] == 0).astype(int)
    d['Brokerage_is_zero'] = (d['Brokerage(S)'] == 0).astype(int)
    # log transform
    num_cols = ['Total_Material_Cost', 'Exwork(M)', 'Freight(O)',
                'Local(Q)', 'Brokerage(S)', 'Total_Import_cost(U)', 'Unit_Price']
    d[num_cols] = np.log1p(d[num_cols])
    # categorical
    for col in ['Vendor_From_Via', 'Vendor_Item', 'Incoterm', 'Item']:
        d[col] = d[col].astype('category')
    return d

df_MM   = apply_fe(df_MM)
df_Thai = apply_fe(df_Thai)

print(f"df_MM rows: {len(df_MM)}, df_Thai rows: {len(df_Thai)}")

df_MM rows: 632, df_Thai rows: 2629


In [52]:
#train

targets = ['Exwork(M)', 'Freight(O)', 'Local(Q)', 'Brokerage(S)']

INCOTERM_RULES = {
    'EXW': [],
    'FCA': ['Exwork(M)'],
    'FOB': ['Exwork(M)'],
    'CFR': ['Exwork(M)', 'Freight(O)'],
    'CIF': ['Exwork(M)', 'Freight(O)'],
    'CPT': ['Exwork(M)', 'Freight(O)'],
    'DAP': ['Exwork(M)', 'Freight(O)', 'Local(Q)'],
    'DDP': ['Exwork(M)', 'Freight(O)', 'Local(Q)', 'Brokerage(S)'],
}

TARGET_ZERO_INCOTERMS = {
    target: [inc for inc, zeros in INCOTERM_RULES.items() if target in zeros]
    for target in targets
}

def train_model_B(df_factory, label):
    X_all = df_factory[['Vendor_From_Via', 'Incoterm', 'Item', 'Total_Material_Cost', 'Unit_Price', 'Qty',
                         'Exwork_is_zero', 'Freight_is_zero', 'Local_is_zero', 'Brokerage_is_zero']]
    _, X_test, _, y_test = train_test_split(X_all, df_factory[targets], test_size=0.2, random_state=42)

    # ----- model1：每個 target 單獨過濾，排除規則強制為 0 的行 -----
    m1 = {}
    for target in targets:
        zero_incoterms = TARGET_ZERO_INCOTERMS[target]
        mask_train = ~df_factory['Incoterm'].isin(zero_incoterms)
        df_trainable = df_factory[mask_train]
        X_t = df_trainable[['Vendor_From_Via', 'Incoterm', 'Item', 'Total_Material_Cost', 'Unit_Price', 'Qty',
                             'Exwork_is_zero', 'Freight_is_zero', 'Local_is_zero', 'Brokerage_is_zero']]
        y_t = df_trainable[target]
        X_tr, _, y_tr, _ = train_test_split(X_t, y_t, test_size=0.2, random_state=42)
        m = xgb.XGBRegressor(enable_categorical=True, random_state=42)
        m.fit(X_tr, y_tr)
        m1[target] = m

    # ----- model2：EXW incoterm 的 Exwork 預測 -----
    mask_exw = (df_factory['Incoterm'] == 'EXW') & (df_factory['Exwork_is_zero'] == 0)
    df_exw = df_factory[mask_exw].copy()
    X_exw = df_exw[['Vendor_Item', 'Total_Material_Cost', 'Vendor_From_Via', 'Item', 'Unit_Price', 'Qty']]
    y_exw = df_exw['Exwork(M)']
    X_tr2, _, y_tr2, _ = train_test_split(X_exw, y_exw, test_size=0.2, random_state=42)
    m2 = xgb.XGBRegressor(enable_categorical=True, random_state=42)
    m2.fit(X_tr2, y_tr2)

    # ----- 評估（Model B 方式）-----
    preds = {}
    for target in targets:
        zero_incoterms = TARGET_ZERO_INCOTERMS[target]
        pred = np.zeros(len(X_test))
        mask_nonzero = ~df_factory.loc[X_test.index, 'Incoterm'].isin(zero_incoterms)
        if target == 'Exwork(M)':
            mask_exw_test = (df_factory.loc[X_test.index, 'Incoterm'] == 'EXW') & \
                            (df_factory.loc[X_test.index, 'Exwork_is_zero'] == 0)
            mask_m1_test  = mask_nonzero & ~mask_exw_test
            if mask_exw_test.sum() > 0:
                X_te_m2 = df_factory.loc[X_test.index[mask_exw_test],
                                         ['Vendor_Item', 'Total_Material_Cost', 'Vendor_From_Via', 'Item', 'Unit_Price', 'Qty']]
                pred[mask_exw_test.values] = np.expm1(m2.predict(X_te_m2)).clip(min=0)
            if mask_m1_test.sum() > 0:
                pred[mask_m1_test.values] = np.expm1(m1[target].predict(X_test[mask_m1_test])).clip(min=0)
        else:
            if mask_nonzero.sum() > 0:
                pred[mask_nonzero.values] = np.expm1(m1[target].predict(X_test[mask_nonzero])).clip(min=0)
        preds[target] = pred

    pred_total = sum(preds[t] for t in targets)
    y_true_total = sum(np.expm1(y_test[t]) for t in targets)

    print(f"\n===== {label} Model B =====")
    for target in targets:
        mask_eval = ~df_factory.loc[X_test.index, 'Incoterm'].isin(TARGET_ZERO_INCOTERMS[target])
        if mask_eval.sum() > 0:
            y_t = np.expm1(y_test.loc[mask_eval, target])
            p_t = preds[target][mask_eval.values]
            print(f"  {target:20s} MAE: {mean_absolute_error(y_t, p_t):>10.2f}  R²: {r2_score(y_t, p_t):.4f}  (n={mask_eval.sum()})")
    print(f"  {'Total_Import_cost':20s} MAE: {mean_absolute_error(y_true_total, pred_total):>10.2f}  R²: {r2_score(y_true_total, pred_total):.4f}")

    return m1, m2

model_MM,   model2_MM   = train_model_B(df_MM,   'MM')
model_Thai, model2_Thai = train_model_B(df_Thai, 'Thailand')


===== MM Model B =====
  Exwork(M)            MAE:      52.48  R²: 0.9882  (n=50)
  Freight(O)           MAE:     154.41  R²: 0.9999  (n=67)
  Local(Q)             MAE:     349.90  R²: 0.6641  (n=127)
  Brokerage(S)         MAE:     399.30  R²: 0.9076  (n=127)
  Total_Import_cost    MAE:     757.81  R²: 0.9981

===== Thailand Model B =====
  Exwork(M)            MAE:      62.04  R²: 0.9296  (n=262)
  Freight(O)           MAE:     337.25  R²: 0.9904  (n=393)
  Local(Q)             MAE:      56.74  R²: 0.9940  (n=518)
  Brokerage(S)         MAE:     106.13  R²: 0.9800  (n=523)
  Total_Import_cost    MAE:     412.40  R²: 0.9915


In [53]:
# test

def test_model(vendor_name, ship_from, ship_via, incoterm, item,
               total_material_cost, unit_price, qty, factory='MM'):
    """
    輸入一行原始數據，輸出各項費用預測。
    factory: 'MM' 或 'Thailand'
    """
    if factory == 'MM':
        _m1, _m2 = model_MM, model2_MM
    else:
        _m1, _m2 = model_Thai, model2_Thai

    ship_from_via   = f"{ship_from}_{ship_via}"
    vendor_from_via = f"{vendor_name}_{ship_from_via}"
    vendor_item     = f"{vendor_name}_{item}"

    zero_cols         = INCOTERM_RULES.get(incoterm, [])
    exwork_is_zero    = 1 if 'Exwork(M)'    in zero_cols else 0
    freight_is_zero   = 1 if 'Freight(O)'   in zero_cols else 0
    local_is_zero     = 1 if 'Local(Q)'     in zero_cols else 0
    brokerage_is_zero = 1 if 'Brokerage(S)' in zero_cols else 0

    input_m1 = pd.DataFrame({
        'Vendor_From_Via':     [vendor_from_via],
        'Incoterm':            [incoterm],
        'Item':                [item],
        'Total_Material_Cost': [np.log1p(total_material_cost)],
        'Unit_Price':          [np.log1p(unit_price)],
        'Qty':                 [qty],
        'Exwork_is_zero':      [exwork_is_zero],
        'Freight_is_zero':     [freight_is_zero],
        'Local_is_zero':       [local_is_zero],
        'Brokerage_is_zero':   [brokerage_is_zero],
    })
    for col in ['Vendor_From_Via', 'Incoterm', 'Item']:
        input_m1[col] = input_m1[col].astype('category')

    results = {}

    # Exwork
    if exwork_is_zero:
        results['Exwork(M)'] = 0.0
    elif incoterm == 'EXW':
        input_m2 = pd.DataFrame({
            'Vendor_Item':         [vendor_item],
            'Total_Material_Cost': [np.log1p(total_material_cost)],
            'Vendor_From_Via':     [vendor_from_via],
            'Item':                [item],
            'Unit_Price':          [np.log1p(unit_price)],
            'Qty':                 [qty],
        })
        for col in ['Vendor_Item', 'Vendor_From_Via', 'Item']:
            input_m2[col] = input_m2[col].astype('category')
        results['Exwork(M)'] = np.expm1(_m2.predict(input_m2))[0].clip(min=0)
    else:
        results['Exwork(M)'] = np.expm1(_m1['Exwork(M)'].predict(input_m1))[0].clip(min=0)

    # Freight, Local, Brokerage
    for target in ['Freight(O)', 'Local(Q)', 'Brokerage(S)']:
        if target in zero_cols:
            results[target] = 0.0
        else:
            results[target] = np.expm1(_m1[target].predict(input_m1))[0].clip(min=0)

    results['Total_Import_cost(U)'] = sum(results[t] for t in ['Exwork(M)', 'Freight(O)', 'Local(Q)', 'Brokerage(S)'])

    print(f"===== 預測結果 [{factory}] =====")
    print(f"  Vendor:    {vendor_name}")
    print(f"  Ship:      {ship_from} → {ship_via}")
    print(f"  Incoterm:  {incoterm}  |  Item: {item}")
    print(f"  Material Cost: {total_material_cost:,.0f}  |  Unit Price: {unit_price:,.2f}  |  Qty: {qty:,}")
    print(f"{'─'*40}")
    print(f"  Exwork(M):          {results['Exwork(M)']:>12,.2f} Baht")
    print(f"  Freight(O):         {results['Freight(O)']:>12,.2f} Baht")
    print(f"  Local(Q):           {results['Local(Q)']:>12,.2f} Baht")
    print(f"  Brokerage(S):       {results['Brokerage(S)']:>12,.2f} Baht")
    print(f"{'─'*40}")
    print(f"  Total Import Cost:  {results['Total_Import_cost(U)']:>12,.2f} Baht")

    return results


# 測試
test_model(
    vendor_name='KINGWHALE CORPORATION',
    ship_from='TAIWAN KEELUNG TO MM',
    ship_via='SEA',
    incoterm='FOB',
    item='CKN',
    total_material_cost=776889,
    unit_price=289.34,
    qty=5370,
    factory='MM'
)

===== 預測結果 [MM] =====
  Vendor:    KINGWHALE CORPORATION
  Ship:      TAIWAN KEELUNG TO MM → SEA
  Incoterm:  FOB  |  Item: CKN
  Material Cost: 776,889  |  Unit Price: 289.34  |  Qty: 5,370
────────────────────────────────────────
  Exwork(M):                  0.00 Baht
  Freight(O):             2,274.44 Baht
  Local(Q):               1,147.68 Baht
  Brokerage(S):           5,256.57 Baht
────────────────────────────────────────
  Total Import Cost:      8,678.68 Baht


{'Exwork(M)': 0.0,
 'Freight(O)': 2274.4384765625,
 'Local(Q)': 1147.6754150390625,
 'Brokerage(S)': 5256.56787109375,
 'Total_Import_cost(U)': 8678.681762695312}

In [54]:
# ===== 批量預測：找最優 ship_via + incoterm 組合 =====

def find_best_combinations(due_date, ship_from, vessel_name, vendor_name, po_num, cust_num,
                           season, buy_no, item, rma_description, um, qty,
                           unit_price, total_mat_cost, ship_via_cat, destination):

    SHIP_VIA_OPTIONS = {
        'SHIP': ['SEA'],
        'AIR':  ['AIR', 'FED', 'DHL'],
    }
    INCOTERM_OPTIONS = ['EXW', 'CIF', 'FOB']

    # 選模型
    if destination == 'MM':
        _m1, _m2 = model_MM, model2_MM
    else:
        _m1, _m2 = model_Thai, model2_Thai

    # 可用組合
    via_options      = SHIP_VIA_OPTIONS.get(ship_via_cat, ['SEA'])
    incoterm_options = ['EXW'] if ship_from.upper().startswith('CHINA') else INCOTERM_OPTIONS

    best_result = None
    best_total  = float('inf')

    for ship_via in via_options:
        for incoterm in incoterm_options:

            zero_cols         = INCOTERM_RULES.get(incoterm, [])
            exwork_is_zero    = 1 if 'Exwork(M)'    in zero_cols else 0
            freight_is_zero   = 1 if 'Freight(O)'   in zero_cols else 0
            local_is_zero     = 1 if 'Local(Q)'     in zero_cols else 0
            brokerage_is_zero = 1 if 'Brokerage(S)' in zero_cols else 0

            ship_from_via   = f"{ship_from}_{ship_via}"
            vendor_from_via = f"{vendor_name}_{ship_from_via}"
            vendor_item     = f"{vendor_name}_{item}"

            input_m1 = pd.DataFrame({
                'Vendor_From_Via':     [vendor_from_via],
                'Incoterm':            [incoterm],
                'Item':                [item],
                'Total_Material_Cost': [np.log1p(total_mat_cost)],
                'Unit_Price':          [np.log1p(unit_price)],
                'Qty':                 [qty],
                'Exwork_is_zero':      [exwork_is_zero],
                'Freight_is_zero':     [freight_is_zero],
                'Local_is_zero':       [local_is_zero],
                'Brokerage_is_zero':   [brokerage_is_zero],
            })
            for col in ['Vendor_From_Via', 'Incoterm', 'Item']:
                input_m1[col] = input_m1[col].astype('category')

            # 預測各項費用
            preds = {}

            if exwork_is_zero:
                preds['Exwork(M)'] = 0.0
            elif incoterm == 'EXW':
                input_m2 = pd.DataFrame({
                    'Vendor_Item':         [vendor_item],
                    'Total_Material_Cost': [np.log1p(total_mat_cost)],
                    'Vendor_From_Via':     [vendor_from_via],
                    'Item':                [item],
                    'Unit_Price':          [np.log1p(unit_price)],
                    'Qty':                 [qty],
                })
                for col in ['Vendor_Item', 'Vendor_From_Via', 'Item']:
                    input_m2[col] = input_m2[col].astype('category')
                preds['Exwork(M)'] = float(np.expm1(_m2.predict(input_m2)).clip(min=0))
            else:
                preds['Exwork(M)'] = float(np.expm1(_m1['Exwork(M)'].predict(input_m1)).clip(min=0))

            for target in ['Freight(O)', 'Local(Q)', 'Brokerage(S)']:
                if target in zero_cols:
                    preds[target] = 0.0
                else:
                    preds[target] = float(np.expm1(_m1[target].predict(input_m1)).clip(min=0))

            total_import = sum(preds.values())

            if total_import < best_total:
                best_total  = total_import
                best_result = {'ship_via': ship_via, 'incoterm': incoterm, **preds,
                               'Total_Import_cost(U)': total_import}

    base = total_mat_cost if total_mat_cost > 0 else 1
    output = pd.DataFrame([{
        'Due Date':                           due_date,
        'Ship From':                          ship_from,
        'Vessel Name':                        vessel_name,
        'Vendor Name':                        vendor_name,
        'PO Num':                             po_num,
        'Cust Num':                           cust_num,
        'Season':                             season,
        'Buy No':                             buy_no,
        'Item':                               item,
        'RMA Description':                    rma_description,
        'UM':                                 um,
        'Qty':                                qty,
        'Unit Price With Surcharge':          unit_price,
        'Total Material Cost (Baht)':         total_mat_cost,
        'Ship Via':                           best_result['ship_via'],
        'Incoterm':                           best_result['incoterm'],
        'Exwork (Baht)':                      round(best_result['Exwork(M)'], 2),
        '%Exwork (M/L)':                      round(best_result['Exwork(M)'] / base, 6),
        'Freight Cost (Baht)':                round(best_result['Freight(O)'], 2),
        '%Freight (O/L)':                     round(best_result['Freight(O)'] / base, 6),
        'Local (Baht)':                       round(best_result['Local(Q)'], 2),
        '%Local (Q/L)':                       round(best_result['Local(Q)'] / base, 6),
        'Brokerage (Baht)':                   round(best_result['Brokerage(S)'], 2),
        '%Brokerage (S/L)':                   round(best_result['Brokerage(S)'] / base, 6),
        'Total Import cost (Baht) (M+O+Q+S)': round(best_result['Total_Import_cost(U)'], 2),
        '% Import Cost (U/L)':                round(best_result['Total_Import_cost(U)'] / base, 6),
    }])

    return output




In [55]:
#test
find_best_combinations(
    due_date        = '2025-01-02',
    ship_from       = 'TAIWAN KEELUNG',
    vessel_name     = 'CAPE SYROS V. 074S ATD (03/12/24)',
    vendor_name     = 'KINGWHALE CORPORATION',
    po_num          = 'M2407CI002',
    cust_num        = 'EMB',
    season          = 'S25',
    buy_no          = 'Lot 2',
    item            = 'CKN',
    rma_description = 'Knit (ผ้าถัก) Fleece',
    um              = 'MT',
    qty             = 5370,
    unit_price      = 289.344,
    total_mat_cost  = 776888.64,
    ship_via_cat    = 'SHIP',
    destination     = 'MM',
)

C:\Users\24503\AppData\Local\Temp\ipykernel_19012\3312957514.py:70: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  preds['Exwork(M)'] = float(np.expm1(_m2.predict(input_m2)).clip(min=0))
C:\Users\24503\AppData\Local\Temp\ipykernel_19012\3312957514.py:78: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  preds[target] = float(np.expm1(_m1[target].predict(input_m1)).clip(min=0))
C:\Users\24503\AppData\Local\Temp\ipykernel_19012\3312957514.py:78: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprec

,Due Date,Ship From,Vessel Name,Vendor Name,PO Num,Cust Num,Season,Buy No,Item,RMA Description,...,Exwork (Baht),%Exwork (M/L),Freight Cost (Baht),%Freight (O/L),Local (Baht),%Local (Q/L),Brokerage (Baht),%Brokerage (S/L),Total Import cost (Baht) (M+O+Q+S),% Import Cost (U/L)
0,2025-01-02,TAIWAN KEELUNG,CAPE SYROS V. 074S ATD (03/12/24),KINGWHALE CORPORATION,M2407CI002,EMB,S25,Lot 2,CKN,Knit (ผ้าถัก) Fleece,...,0.0,0.0,0.0,0.0,2462.19,0.003169,6701.83,0.008626,9164.02,0.011796


In [56]:
import joblib
import json
import pickle

# 保存模型
joblib.dump(models, 'models.pkl')  # 主模型 (Exwork, Freight, Local, Brokerage)
joblib.dump(model2, 'model2.pkl')  # EXW 专用模型

# 保存 INCOTERM_RULES
with open('incoterm_rules.json', 'w', encoding='utf-8') as f:
    json.dump(INCOTERM_RULES, f, ensure_ascii=False)

# 保存需要的列名和数据
with open('model_metadata.pkl', 'wb') as f:
    pickle.dump({
        'features': ['Vendor_From_Via', 'Incoterm', 'Item', 'Total_Material_Cost', 
                     'Unit_Price', 'Exwork_is_zero', 'Freight_is_zero', 
                     'Local_is_zero', 'Brokerage_is_zero'],
        'targets': ['Exwork(M)', 'Freight(O)', 'Local(Q)', 'Brokerage(S)'],
        'incoterm_rules': INCOTERM_RULES
    }, f)

print("✅ 模型保存成功！")

✅ 模型保存成功！
